# DỰ ÁN 1 - XÂY DỰNG MÔ HÌNH PHÂN LOẠI VÀ DỰ BÁO RỦI RO KHÁCH HÀNG VAY VỐN

**Notebook 02/07 - Database Pipeline trung tâm (PostgreSQL)**

---

**Mục tiêu:** Dựng PostgreSQL thành nguồn dữ liệu trung tâm của dự án. Notebook đóng vai trò bản diễn giải ngắn gọn: chạy tới đâu thì các script trong `sql/` tự động chạy tới đó, theo đúng thứ tự phụ thuộc.

**Input:** `data/raw/*.csv` (8 bảng thô đã khảo sát ở NB01); các script `sql/01_create_tables.sql` đến `sql/05_indexes.sql`.

**Output:** Database PostgreSQL gồm 8 bảng raw, 7 view nghiệp vụ và 3 materialized view tổng hợp — kèm hợp đồng dữ liệu để NB03 đến NB07 và `app/` đọc bằng `pd.read_sql`.

**Pipeline:** 01. Data Understanding -> **02. Database Pipeline** -> NB03/NB04/NB05/NB06 đọc dữ liệu từ PostgreSQL khi cần.

**Nối tiếp NB01:** NB01 dừng ở mức khảo sát, chưa biến đổi dữ liệu, và kết lại bằng bốn thách thức của bộ dữ liệu (mục 8.1 và 9.1 của NB01). NB02 là bước đầu tiên thực sự xử lý dữ liệu, và nhận trách nhiệm với **thách thức 1 (phân mảnh, quan hệ 1-n lồng 3 tầng)** và **thách thức 2 (quy mô lớn)**. Hai thách thức còn lại thuộc NB03 và NB06.

## Chuẩn bị trước khi chạy

Cần chuẩn bị sẵn trên từng máy:

1. Cài thư viện: `pip install -r requirements.txt`.
2. Bật PostgreSQL và tạo database rỗng, ví dụ `CREATE DATABASE credit_risk_db;`.
3. Tạo file `.env` ở thư mục gốc repo (chép từ `.env.example`) và điền đúng thông số máy mình.
4. Đặt 8 file CSV thô vào `data/raw/`.

Notebook này **chỉ cần `data/raw/`**, không phụ thuộc `data/processed/`, nên chạy được ngay sau NB01 mà không phải chờ NB03 hay NB05.

Chạy lại notebook nhiều lần vẫn an toàn: script tạo bảng có `DROP TABLE IF EXISTS`, bước import raw có `TRUNCATE` trước `COPY`. Ai đã tạo bảng và import bằng tay trên pgAdmin thì chạy notebook vẫn không hỏng gì, dữ liệu chỉ được nạp lại từ đầu.

## 1. Vai trò của NB02 trong pipeline

### 1.1. NB01 để lại gì cho NB02

NB01 khảo sát dữ liệu và kết lại bằng bốn thách thức (mục 8.1 và 9.1 của NB01). Bảng dưới đối chiếu từng thách thức với nơi xử lý, để thấy rõ NB02 nhận phần nào:

| Thách thức NB01 nêu | Số liệu NB01 đo được | Xử lý ở đâu |
|---|---|---|
| 1. Phân mảnh, quan hệ 1-n lồng 3 tầng | 307.511 khách -> 1.716.428 khoản vay ngoài -> 27.299.925 dòng số dư theo tháng | **NB02** — gom bảng phụ về mức khách hàng bằng SQL (mục 5) |
| 2. Quy mô lớn | 8 bảng, khoảng 2,5 GB; riêng `bureau_balance` 27,3 triệu dòng | **NB02** — để database xử lý theo khối, không kéo vào RAM Python (mục 4, mục 5) |
| 3. Mất cân bằng lớp | 24.825 khách `TARGET=1` (8,07%) so với 282.686 khách `TARGET=0` | NB06 — chọn chỉ số đánh giá và ngưỡng quyết định |
| 4. Dữ liệu thật, chưa làm sạch | 67/122 cột có ô thiếu; `DAYS_EMPLOYED = 365243` xuất hiện 55.374 lần | NB03 — làm sạch kèm giải trình thống kê |

Các con số ở cột giữa được dùng lại làm mốc đối chiếu ở mục 7: sau khi import, số dòng trong database phải khớp đúng con số NB01 đã đo trên file CSV.

### 1.2. Bảng nào do notebook nào tạo

NB02 gom toàn bộ phần database của dự án về một chỗ. Các notebook sau không tự đoán tên bảng và không viết lại các phép join lớn nếu dữ liệu đó đã có sẵn trong database.

| Loại dữ liệu | Nơi tạo | Notebook đọc tiếp |
|---|---|---|
| Bảng raw (`application_train`, `bureau`, ...) | **NB02** qua `sql/01` và `COPY` | NB03, NB04, NB05 |
| View nghiệp vụ (`v_*`) | **NB02** qua `sql/03_views.sql` | NB04, NB05 |
| Bảng tổng hợp (`agg_*`) | **NB02** qua `sql/04_aggregation.sql` | NB04, NB05 |
| Bảng clean (`application_train_clean`, `application_test_clean`) | NB03 ghi ngược về DB | NB04, NB05, NB06 |
| Bảng đặc trưng (`train_features`, `test_features`) | NB05 ghi ngược về DB | NB06, NB07, `app/` |

Hai dòng cuối **không thuộc phạm vi NB02**: theo sơ đồ pipeline chốt ở NB01 mục 8.2, dữ liệu do Python tạo ra thì chính notebook tạo ra nó chịu trách nhiệm ghi lại database. NB02 chỉ công bố tên bảng để cả nhóm dùng thống nhất (mục 6).

### 1.3. Vì sao chia việc giữa SQL và Python

Nguyên tắc phân công (thử thách số 6 của đề bài):

- **SQL làm:** join nhiều bảng, aggregation theo khách hàng, filtering, indexing. Đây là việc chạy trên hàng chục triệu dòng, để database làm thì nhanh hơn và không tốn RAM máy cá nhân.
- **Python làm:** feature engineering, làm sạch dựa trên thống kê, huấn luyện mô hình. Đây là việc cần thư viện khoa học dữ liệu mà SQL không có.
- **Kết quả của Python quay lại SQL:** bảng clean (NB03) và bảng đặc trưng (NB05) được chính notebook sinh ra chúng ghi ngược về database, nhờ vậy database là nguồn trung tâm duy nhất chứ không chỉ là kho chứa dữ liệu thô.

## 2. Chuẩn bị môi trường và kết nối database

### 2.1. Nạp thư viện và đọc cấu hình kết nối

Đoạn code bên dưới nạp thư viện, đọc thông số database từ `.env` và xác định các thư mục của repo.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv, find_dotenv

# find_dotenv() dò ngược lên các thư mục cha để tìm .env, nhờ vậy notebook chạy
# được dù kernel mở từ notebooks/ hay từ thư mục gốc repo.
# override=True buộc nạp lại giá trị mới nhất trong .env, tránh dùng nhầm biến
# môi trường cũ còn sót trong kernel (nguyên nhân lỗi xác thực khi chưa restart).
load_dotenv(find_dotenv(), override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "credit_risk_db")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

# Xác định thư mục gốc repo bằng cách tìm thư mục chứa cả sql/ và data/,
# thay vì viết cứng '../' (viết cứng sẽ hỏng nếu chạy từ thư mục khác).
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "sql").is_dir() and (p / "data").is_dir()
)
SQL_DIR = REPO_ROOT / "sql"
DATA_RAW = REPO_ROOT / "data" / "raw"

print("Thông số kết nối database:")
print(f"- Host: {db_host}")
print(f"- Port: {db_port}")
print(f"- Database: {db_name}")
print(f"- User: {db_user}")
print(f"\nThư mục gốc repo: {REPO_ROOT}")

**Nhận xét:** Mật khẩu và tên database không viết thẳng trong notebook mà đọc từ `.env` (file này đã được gitignore). Nhờ vậy mỗi máy dùng cấu hình riêng mà notebook vẫn commit chung được, và mật khẩu không bị đẩy lên GitHub.

### 2.2. Hàm chạy một file SQL

Đoạn code bên dưới định nghĩa hàm đọc một file trong `sql/` và thực thi toàn bộ nội dung file đó.

In [ ]:
def execute_sql_file(file_path, conn):
    """Đọc và thực thi toàn bộ câu lệnh trong một file SQL."""
    start_time = time.time()
    with open(file_path, "r", encoding="utf-8") as f:
        sql_script = f.read()

    try:
        with conn.cursor() as cursor:
            cursor.execute(sql_script)
        conn.commit()
        print(f"Đã chạy file: {Path(file_path).name} ({time.time() - start_time:.2f} giây).")
    except Exception:
        # Rollback để kết nối không kẹt ở trạng thái lỗi. Nếu không rollback,
        # mọi cell sau đều hỏng theo với thông báo khó hiểu
        # "current transaction is aborted".
        conn.rollback()
        raise

**Nhận xét:** Đây là hàm giúp notebook đóng vai trò bản diễn giải: nội dung SQL vẫn nằm nguyên trong `sql/`, notebook chỉ gọi chạy theo đúng thứ tự. Sửa logic SQL thì sửa trong file `.sql`, không phải sửa notebook — tránh tình trạng hai nơi giữ hai phiên bản khác nhau.

### 2.3. Mở kết nối tới PostgreSQL

Đoạn code bên dưới mở kết nối và in phiên bản PostgreSQL để xác nhận kết nối thành công.

In [ ]:
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    database=db_name,
    user=db_user,
    password=db_password,
)

with conn.cursor() as cursor:
    cursor.execute("SELECT version()")
    print("Kết nối thành công:", cursor.fetchone()[0].split(",")[0])

**Nhận xét:** Kết nối mở thành công nghĩa là PostgreSQL đang chạy, database đã tồn tại và thông số trong `.env` khớp. Nếu cell này lỗi thì các mục sau không cần chạy — phải sửa `.env` hoặc bật service PostgreSQL trước.

## 3. Tạo bảng raw trong PostgreSQL

Chạy `sql/01_create_tables.sql` để tạo 8 bảng thô. Script có `DROP TABLE IF EXISTS ... CASCADE` ở đầu nên chạy lại được nhiều lần mà không xung đột.

### 3.1. Chạy script tạo bảng và kiểm tra đã đủ 8 bảng

Đoạn code bên dưới chạy `sql/01_create_tables.sql` rồi đối chiếu danh sách bảng vừa tạo.

In [ ]:
execute_sql_file(SQL_DIR / "01_create_tables.sql", conn)

expected_tables = [
    "application_train", "application_test", "bureau", "bureau_balance",
    "previous_application", "pos_cash_balance", "installments_payments",
    "credit_card_balance",
]

with conn.cursor() as cursor:
    cursor.execute(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
        """
    )
    actual_tables = {row[0] for row in cursor.fetchall()}

# Kiểm tra ngay tại đây thay vì để bước import phía sau báo lỗi khó hiểu.
missing_tables = [t for t in expected_tables if t not in actual_tables]
assert not missing_tables, f"Thiếu bảng sau khi chạy sql/01: {missing_tables}"
print(f"Đã tạo đủ {len(expected_tables)} bảng raw.")

**Nhận xét:** Schema trong `sql/01_create_tables.sql` khai 25 cột kiểu số thực là `DOUBLE PRECISION` thay vì `INT`, vì dữ liệu Home Credit lưu các cột này dưới dạng số thực (`-3648.0`, `12.0`). Khai `INT` sẽ làm `COPY` dừng với lỗi `invalid input syntax for type integer`.

## 4. Import dữ liệu raw vào database

Dùng `COPY ... FROM STDIN` thay vì `COPY ... FROM 'đường/dẫn'`. Khác biệt quan trọng:

- `COPY FROM 'đường/dẫn'` là **server-side**: tiến trình PostgreSQL tự mở file, nên file phải nằm trên máy chạy server và tài khoản service phải có quyền đọc. Đây là nguyên nhân lỗi phân quyền khi chạy `sql/02_import_data.sql` trực tiếp trên pgAdmin.
- `COPY FROM STDIN` là **client-side**: Python đọc file rồi đẩy luồng dữ liệu qua kết nối, nên chạy được với đường dẫn tương đối trong repo và không phụ thuộc quyền của service PostgreSQL.

### 4.1. Hàm import một file CSV bằng COPY FROM STDIN

Đoạn code bên dưới định nghĩa hàm import một file CSV vào bảng tương ứng.

In [ ]:
def import_csv_to_db(table_name, csv_path, conn):
    """Import một file CSV vào bảng đã có sẵn bằng COPY FROM STDIN.

    TRUNCATE ngay trước COPY để hàm chạy lại được nhiều lần mà không nạp
    trùng dữ liệu. Hai lệnh nằm chung một transaction: nếu COPY lỗi thì
    rollback cả hai, bảng giữ nguyên dữ liệu cũ chứ không rơi vào trạng
    thái rỗng dở dang.
    """
    start_time = time.time()
    print(f"Đang import '{table_name}' từ {Path(csv_path).name} ...")

    copy_query = sql.SQL(
        "COPY {} FROM STDIN WITH (FORMAT CSV, HEADER, DELIMITER ',')"
    ).format(sql.Identifier(table_name))

    try:
        with conn.cursor() as cursor:
            cursor.execute(
                sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY CASCADE").format(
                    sql.Identifier(table_name)
                )
            )
            with open(csv_path, "r", encoding="utf-8") as f:
                cursor.copy_expert(copy_query, f)
        conn.commit()
        print(f"  Xong trong {time.time() - start_time:.2f} giây.")
    except Exception:
        conn.rollback()
        raise

**Nhận xét:** `TRUNCATE` nằm bên trong hàm chứ không tách ra ngoài là có chủ đích: nó buộc mỗi lần import phải xóa sạch bảng trước, nên không có đường nào chạy `COPY` mà quên xóa dữ liệu cũ. Đây là lỗi nạp trùng dữ liệu nhóm từng gặp ở bản NB02 trước.

### 4.2. Import 8 bảng raw

Đoạn code bên dưới import lần lượt 8 file CSV thô vào database.

In [ ]:
# Tên bảng trong DB không trùng tên file ở trường hợp POS_CASH_balance.csv ->
# pos_cash_balance, nên phải khai báo mapping tường minh thay vì suy từ tên file.
csv_mapping = {
    "application_train": DATA_RAW / "application_train.csv",
    "application_test": DATA_RAW / "application_test.csv",
    "bureau": DATA_RAW / "bureau.csv",
    "bureau_balance": DATA_RAW / "bureau_balance.csv",
    "previous_application": DATA_RAW / "previous_application.csv",
    "pos_cash_balance": DATA_RAW / "POS_CASH_balance.csv",
    "installments_payments": DATA_RAW / "installments_payments.csv",
    "credit_card_balance": DATA_RAW / "credit_card_balance.csv",
}

# Thiếu file thì dừng ngay, không import nửa vời rồi mới phát hiện ở cuối.
missing_csv = [str(p) for p in csv_mapping.values() if not p.exists()]
assert not missing_csv, f"Thiếu file CSV thô: {missing_csv}"

total_start = time.time()
for table, path in csv_mapping.items():
    import_csv_to_db(table, path, conn)

print(f"\nHoàn thành import 8 bảng raw trong {time.time() - total_start:.2f} giây.")

**Nhận xét:** Toàn bộ dữ liệu thô đã nằm trong PostgreSQL. Hai bảng lớn nhất là `bureau_balance` (khoảng 27,3 triệu dòng) và `installments_payments` (khoảng 13,6 triệu dòng) chiếm gần hết thời gian import. Chính quy mô này là lý do các phép gom nhóm trên hai bảng đó phải để SQL làm thay vì kéo hết về pandas.

## 5. Tạo view, join, aggregation và index

Đây là phần trung tâm của notebook. Ba script chạy theo đúng thứ tự phụ thuộc:

- `sql/03_views.sql` — 7 view làm sạch và tính chỉ số nghiệp vụ (tỷ lệ dư nợ trên hạn mức, số ngày trễ hạn, số tiền đóng thiếu, tỷ lệ dùng thẻ...).
- `sql/04_aggregation.sql` — 3 materialized view gom lịch sử trả góp, POS/Cash và thẻ tín dụng về **một dòng cho mỗi khách hàng**, để join vào bảng chính mà không nhân dòng.
- `sql/05_indexes.sql` — index khóa ngoại, composite index và partial index để tăng tốc join và lọc.

### 5.1. Chạy script tạo view, aggregation và index

Đoạn code bên dưới chạy ba script `sql/03`, `sql/04` và `sql/05` theo thứ tự.

In [ ]:
for script_name in ["03_views.sql", "04_aggregation.sql", "05_indexes.sql"]:
    execute_sql_file(SQL_DIR / script_name, conn)

print("\nĐã tạo xong view, bảng tổng hợp và index.")

**Nhận xét:** Thứ tự chạy không đổi được. `sql/04` gom nhóm dữ liệu từ các bảng raw nên phải có bảng trước; `sql/05` đánh index trên cả bảng raw lẫn kết quả tổng hợp nên phải chạy sau cùng. Đảo thứ tự sẽ lỗi vì đối tượng chưa tồn tại.

### 5.2. Xem thử kết quả join giữa bảng chính và bảng tổng hợp

Đoạn code bên dưới join `application_train` với ba bảng tổng hợp và lấy thử vài dòng để kiểm chứng phép join chạy đúng.

In [ ]:
# LEFT JOIN chứ không phải INNER JOIN: nhiều khách hàng không có lịch sử trả góp
# hoặc chưa từng dùng thẻ tín dụng. Dùng INNER JOIN sẽ âm thầm làm mất các
# khách hàng này khỏi tập huấn luyện.
join_preview_query = """
    SELECT
        a.sk_id_curr,
        a.target,
        a.amt_credit,
        i.cnt_late_payments,
        i.avg_days_late,
        p.max_pos_cash_dpd,
        c.avg_card_balance
    FROM application_train AS a
    LEFT JOIN agg_installments_summary AS i ON a.sk_id_curr = i.sk_id_curr
    LEFT JOIN agg_pos_cash_summary     AS p ON a.sk_id_curr = p.sk_id_curr
    LEFT JOIN agg_credit_card_summary  AS c ON a.sk_id_curr = c.sk_id_curr
    ORDER BY a.sk_id_curr
    LIMIT 10
"""
join_preview = pd.read_sql(join_preview_query, conn)
display(join_preview)

# Số dòng sau join phải bằng đúng số dòng bảng gốc. Lớn hơn nghĩa là một
# khóa bị lặp ở bảng tổng hợp và phép join đã nhân dòng.
with conn.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM application_train")
    rows_before = cursor.fetchone()[0]
    cursor.execute("""
        SELECT COUNT(*)
        FROM application_train AS a
        LEFT JOIN agg_installments_summary AS i ON a.sk_id_curr = i.sk_id_curr
        LEFT JOIN agg_pos_cash_summary     AS p ON a.sk_id_curr = p.sk_id_curr
        LEFT JOIN agg_credit_card_summary  AS c ON a.sk_id_curr = c.sk_id_curr
    """)
    rows_after = cursor.fetchone()[0]

print(f"Số dòng trước join: {rows_before:,}")
print(f"Số dòng sau join:   {rows_after:,}")
assert rows_after == rows_before, "Phép join đã nhân dòng — kiểm tra lại khóa của các bảng agg_*."

**Nhận xét:** Số dòng trước và sau join bằng nhau, tức các bảng `agg_*` giữ đúng một dòng cho mỗi khách hàng và phép join không nhân dòng. Đây là kịch bản unit test mà đề bài yêu cầu ở Chương 2 whitepaper — nhân dòng là lỗi nguy hiểm nhất khi join quan hệ một-nhiều, vì nó không báo lỗi mà chỉ lặng lẽ làm sai mọi thống kê phía sau. Các ô trống trong kết quả là khách hàng chưa từng có lịch sử ở bảng phụ, đúng như mong đợi của `LEFT JOIN`.

## 6. Hợp đồng dữ liệu cho các notebook sau

Đây là danh sách tên bảng và view mà cả nhóm dùng thống nhất. Notebook sau chỉ cần đọc theo đúng tên này, không tự đặt tên khác và không viết lại các phép join lớn đã làm ở mục 5.

### 6.1. Danh sách bảng và view kèm câu truy vấn

Đoạn code bên dưới liệt kê bảng/view kèm notebook sử dụng và câu truy vấn tương ứng.

In [ ]:
# Cột "Do ai tạo" phân định trách nhiệm: NB02 tạo các đối tượng ở nhóm trên,
# hai nhóm dưới do NB03 và NB05 tự ghi ngược về database.
DATASETS_FOR_NOTEBOOKS = [
    ["application_train", "bảng raw", "NB02", "NB03, NB05", "SELECT * FROM application_train"],
    ["application_test", "bảng raw", "NB02", "NB03, NB05, NB07, app", "SELECT * FROM application_test"],
    ["v_application_all", "view", "NB02", "NB03, NB04", "SELECT * FROM v_application_all"],
    ["v_bureau_clean", "view", "NB02", "NB04, NB05", "SELECT * FROM v_bureau_clean"],
    ["v_previous_application_clean", "view", "NB02", "NB04, NB05", "SELECT * FROM v_previous_application_clean"],
    ["v_bureau_balance_summary", "view", "NB02", "NB04, NB05", "SELECT * FROM v_bureau_balance_summary"],
    ["agg_installments_summary", "materialized view", "NB02", "NB04, NB05", "SELECT * FROM agg_installments_summary"],
    ["agg_pos_cash_summary", "materialized view", "NB02", "NB04, NB05", "SELECT * FROM agg_pos_cash_summary"],
    ["agg_credit_card_summary", "materialized view", "NB02", "NB04, NB05", "SELECT * FROM agg_credit_card_summary"],
    ["application_train_clean", "bảng clean", "NB03", "NB04, NB05", "SELECT * FROM application_train_clean"],
    ["application_test_clean", "bảng clean", "NB03", "NB05, NB07", "SELECT * FROM application_test_clean"],
    ["train_features", "bảng đặc trưng", "NB05", "NB06", "SELECT * FROM train_features"],
    ["test_features", "bảng đặc trưng", "NB05", "NB07, app", "SELECT * FROM test_features"],
]

contract_df = pd.DataFrame(
    DATASETS_FOR_NOTEBOOKS,
    columns=["Tên bảng/view", "Loại", "Do ai tạo", "Notebook đọc", "Câu truy vấn"],
)
contract_df

**Nhận xét:** Chín dòng đầu là các đối tượng NB02 tạo ra và đã kiểm tra ở mục 7, dùng được ngay. Bốn dòng cuối phụ thuộc NB03 và NB05 — tới thời điểm viết notebook này, NB03 đã có sẵn code ghi ngược nhưng chưa chạy, còn NB05 thì chưa có. Ai cần bốn bảng đó phải kiểm tra sự tồn tại trước khi đọc, thay vì mặc định là đã có:

```python
with conn.cursor() as cursor:
    cursor.execute("SELECT to_regclass('train_features')")
    assert cursor.fetchone()[0] is not None, "Chưa có bảng train_features — cần chạy NB05 trước."
```

## 7. Kiểm tra dữ liệu sau import và sau aggregation

Kiểm tra chia làm hai nhóm, mỗi nhóm bắt một loại rủi ro khác nhau:

1. Số dòng 8 bảng raw phải khớp đúng con số NB01 đã đo ở mục 4 của notebook đó — bắt lỗi import thiếu dòng hoặc nạp trùng dữ liệu.
2. View và materialized view phải tồn tại, và bảng tổng hợp phải giữ đúng một dòng cho mỗi khách hàng — bắt lỗi nhân dòng khi join.

### 7.1. Đối chiếu số dòng 8 bảng raw

Đoạn code bên dưới đối chiếu số dòng thực tế trong database với số dòng đã xác định ở NB01.

In [ ]:
# Số dòng chuẩn lấy từ NB01 (Data Understanding).
expected_raw_rows = {
    "application_train": 307_511,
    "application_test": 48_744,
    "bureau": 1_716_428,
    "bureau_balance": 27_299_925,
    "previous_application": 1_670_214,
    "pos_cash_balance": 10_001_358,
    "installments_payments": 13_605_401,
    "credit_card_balance": 3_840_312,
}

raw_checks = []
with conn.cursor() as cursor:
    for table_name, expected in expected_raw_rows.items():
        cursor.execute(sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(table_name)))
        actual = cursor.fetchone()[0]
        raw_checks.append({
            "Bảng raw": table_name,
            "Số dòng mong đợi": expected,
            "Số dòng thực tế": actual,
            "Kết quả": "OK" if actual == expected else "LỆCH",
        })

raw_check_df = pd.DataFrame(raw_checks)
raw_check_df

**Nhận xét:** Cả 8 bảng khớp đúng số dòng của NB01, nghĩa là bước `COPY` không bỏ sót dòng nào và cũng không nạp trùng. Nếu một bảng có số dòng gấp đôi mong đợi thì gần như chắc chắn `TRUNCATE` đã không chạy trước `COPY`.

### 7.2. Kiểm tra view và khóa sau aggregation

Đoạn code bên dưới kiểm tra sự tồn tại của view, materialized view và tính duy nhất của khóa trong bảng tổng hợp.

In [ ]:
object_names = [
    "v_application_all", "v_bureau_clean", "v_bureau_balance_summary",
    "v_previous_application_clean", "v_installments_detail", "v_pos_cash_clean",
    "v_credit_card_clean", "agg_installments_summary", "agg_pos_cash_summary",
    "agg_credit_card_summary",
]

with conn.cursor() as cursor:
    object_checks = []
    for object_name in object_names:
        # to_regclass trả về NULL nếu đối tượng không tồn tại, nên kiểm tra được
        # cả view lẫn materialized view chỉ bằng một câu truy vấn.
        cursor.execute("SELECT to_regclass(%s)", (object_name,))
        exists = cursor.fetchone()[0] is not None
        row_count = None
        if exists:
            cursor.execute(sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(object_name)))
            row_count = cursor.fetchone()[0]
        object_checks.append({
            "Bảng/View": object_name,
            "Tồn tại": exists,
            "Số dòng": row_count,
        })

    # Bảng tổng hợp phải giữ đúng một dòng cho mỗi khách hàng. Số dòng lớn hơn
    # số khách duy nhất nghĩa là phép join phía sau sẽ nhân dòng.
    unique_checks = []
    for object_name in ["agg_installments_summary", "agg_pos_cash_summary", "agg_credit_card_summary"]:
        cursor.execute(
            sql.SQL("SELECT COUNT(*), COUNT(DISTINCT sk_id_curr) FROM {}").format(
                sql.Identifier(object_name)
            )
        )
        row_count, distinct_customer = cursor.fetchone()
        unique_checks.append({
            "Bảng tổng hợp": object_name,
            "Số dòng": row_count,
            "Số khách duy nhất": distinct_customer,
            "Kết quả": "OK" if row_count == distinct_customer else "BỊ NHÂN DÒNG",
        })

print("Kiểm tra view và materialized view:")
display(pd.DataFrame(object_checks))

print("\nKiểm tra khóa sau aggregation:")
unique_check_df = pd.DataFrame(unique_checks)
unique_check_df

**Nhận xét:** Số dòng của ba bảng `agg_*` bằng đúng số khách hàng duy nhất, xác nhận `GROUP BY sk_id_curr` đã gom đúng. Số dòng các bảng này (khoảng 340 nghìn) nhỏ hơn rất nhiều so với bảng nguồn (13,6 triệu dòng `installments_payments`) — đây chính là giá trị của việc đẩy aggregation sang SQL: notebook sau chỉ phải đọc phần đã gọn.

### 7.3. Chốt kết quả kiểm tra bằng assert

Đoạn code bên dưới dùng `assert` để bắt lỗi phải nổ ra thay vì trôi qua.

In [ ]:
# Dùng assert thay vì chỉ in bảng: nếu số liệu sai mà notebook vẫn chạy hết thì
# người đọc dễ tưởng mọi thứ đều ổn. Đây là bài học nhóm rút ra từ NB03 và NB05.
failed_raw = raw_check_df.loc[raw_check_df["Kết quả"] != "OK", "Bảng raw"].tolist()
assert not failed_raw, f"Số dòng bảng raw không khớp NB01: {failed_raw}"

failed_unique = unique_check_df.loc[unique_check_df["Kết quả"] != "OK", "Bảng tổng hợp"].tolist()
assert not failed_unique, f"Bảng tổng hợp bị nhân dòng: {failed_unique}"

missing_objects = [
    row["Bảng/View"] for row in object_checks if not row["Tồn tại"]
]
assert not missing_objects, f"Thiếu view hoặc bảng tổng hợp: {missing_objects}"

print("Toàn bộ kiểm tra đã qua: 8 bảng raw đúng số dòng, 10 view/bảng tổng hợp tồn tại, "
      "không có bảng nào bị nhân dòng.")

**Nhận xét:** Ba câu `assert` biến các bảng kiểm tra phía trên thành điều kiện bắt buộc. Notebook chạy hết mà không dừng ở đây thì dữ liệu trong database đã đúng; ngược lại kernel dừng ngay tại dòng sai và in rõ bảng nào có vấn đề, không để lỗi trôi qua trong im lặng.

## 8. Đóng kết nối database

### 8.1. Đóng kết nối

Đoạn code bên dưới đóng kết nối tới PostgreSQL.

In [ ]:
conn.close()
print("Đã đóng kết nối PostgreSQL.")

**Nhận xét:** Đóng kết nối để PostgreSQL giải phóng phiên làm việc. Kernel Jupyter chạy lâu mà không đóng sẽ để lại nhiều kết nối treo, dần chạm giới hạn `max_connections` của server và các notebook khác không kết nối được nữa.

## 9. Tổng kết

### 9.1. Những việc notebook này đã làm

1. Tạo 8 bảng raw theo schema ở `sql/01_create_tables.sql`.
2. Import toàn bộ CSV thô bằng `COPY FROM STDIN` — chạy được với đường dẫn tương đối và không vướng phân quyền như `COPY` server-side.
3. Tạo 7 view nghiệp vụ, 3 materialized view tổng hợp và bộ index qua `sql/03` đến `sql/05`.
4. Kiểm chứng phép join giữa bảng chính và ba bảng tổng hợp không làm nhân dòng.
5. Công bố hợp đồng dữ liệu để NB03 đến NB07 và `app/` đọc chung một nguồn.
6. Kiểm tra hai tầng: số dòng raw khớp NB01, và khóa sau aggregation là duy nhất.

### 9.2. Đối chiếu lại hai thách thức NB02 nhận từ NB01

Ở mục 1.1, NB02 nhận trách nhiệm với hai trong bốn thách thức mà NB01 nêu. Kết quả:

| Thách thức | NB01 đo được | NB02 giải quyết thế nào | Kiểm chứng tại |
|---|---|---|---|
| 1. Phân mảnh, quan hệ 1-n lồng 3 tầng | 307.511 khách -> 1.716.428 khoản vay ngoài -> 27.299.925 dòng số dư | 3 materialized view gom lịch sử trả góp, POS/Cash và thẻ tín dụng về đúng **một dòng cho mỗi khách hàng**, ghép được vào bảng chính mà không nhân dòng | Mục 5.2, mục 7.2 |
| 2. Quy mô lớn | 8 bảng, khoảng 2,5 GB; `bureau_balance` 27,3 triệu dòng | Toàn bộ gom nhóm chạy trong database theo khối, notebook sau chỉ đọc phần đã gọn (khoảng 340 nghìn dòng thay vì 13,6 triệu) | Mục 5.1, mục 7.2 |

Hai thách thức còn lại của NB01 — mất cân bằng lớp và dữ liệu chưa làm sạch — không xử lý được bằng SQL, nên thuộc NB03 (làm sạch có chủ đích) và NB06 (chọn chỉ số đánh giá và ngưỡng quyết định).

### 9.3. Điều rút ra và bước tiếp theo

**Điều rút ra:** phần việc nặng về dữ liệu lớn (gom 27,3 triệu dòng `bureau_balance` và 13,6 triệu dòng `installments_payments`) đặt ở SQL vì database xử lý theo khối và không tốn RAM máy cá nhân; phần việc cần thư viện khoa học dữ liệu (làm sạch theo thống kê, tạo đặc trưng, huấn luyện mô hình) đặt ở Python. Ranh giới này chính là yêu cầu chia pipeline hai phần của đề bài.

**Bước tiếp theo:** NB03 làm sạch có chủ đích trên `application_train` và `application_test`, rồi ghi bảng clean ngược về database. NB04 dùng view và bảng tổng hợp tạo ở đây để trực quan hóa.